# CAIR Lab — Twitter Entity Sentiment

In [1]:
import os
import glob
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
SUB_DIR = 'submissions'
os.makedirs(SUB_DIR, exist_ok=True)

train_df = pd.read_csv('CAIRLAB.csv', header=None, names=['ID', 'Entity', 'Sentiment', 'Text'])
test_df = pd.read_csv('test.csv')

train_df['Text'] = train_df['Text'].fillna('')
test_df['Text'] = test_df['Text'].fillna('')
train_df = train_df[train_df['Text'].str.strip() != ''].reset_index(drop=True)
print('train:', train_df.shape, '| test:', test_df.shape)

train: (67888, 4) | test: (1000, 3)


## Данные

In [2]:
print('test ID в train:', test_df['ID'].isin(train_df['ID']).mean())
print('строк на ID:', train_df.groupby('ID').size().value_counts().to_dict())
print('сущностей:', train_df['Entity'].nunique(), '| новых сущностей в test:', (~test_df['Entity'].isin(train_df['Entity'])).sum())
train_df['Sentiment'].value_counts(normalize=True).round(3)

test ID в train: 0.0
строк на ID: {6: 11145, 2: 159, 5: 128, 4: 15}
сущностей: 32 | новых сущностей в test: 0


Sentiment
Negative      0.305
Positive      0.280
Neutral       0.241
Irrelevant    0.174
Name: proportion, dtype: float64

In [3]:
train_df.loc[train_df['ID'] == 2401, ['Sentiment', 'Text']]

,Sentiment,Text
0,Positive,im getting on borderlands and i will murder yo...
1,Positive,I am coming to the borders and I will kill you...
2,Positive,im getting on borderlands and i will kill you ...
3,Positive,im coming on borderlands and i will murder you...
4,Positive,im getting on borderlands 2 and i will murder ...
5,Positive,im getting into borderlands and i can murder y...


## Валидация

In [4]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score

X = train_df['Entity'] + ' ' + train_df['Text']
y = train_df['Sentiment']
X_test = test_df['Entity'] + ' ' + test_df['Text']
LABELS = set(y.unique())

folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
            .split(X, y, groups=train_df['ID']))

def take(data, idx):
    return data.iloc[idx] if hasattr(data, 'iloc') else data[idx]

def cv_score(model, X=X):
    scores = []
    for tr, va in folds:
        model.fit(take(X, tr), y.iloc[tr])
        scores.append(accuracy_score(y.iloc[va], model.predict(take(X, va))))
    return np.mean(scores), np.std(scores)

results = {}

def make_submission(model, name, X=X, X_test=X_test):
    cv, std = cv_score(model, X)
    model.fit(X, y)
    sub = pd.DataFrame({'ID': test_df['ID'], 'Sentiment': model.predict(X_test)})

    assert len(sub) == len(test_df) and sub['ID'].is_unique
    assert set(sub['Sentiment']) <= LABELS

    path = f'{SUB_DIR}/sub_{name}_cv{cv:.4f}.csv'
    sub.to_csv(path, index=False)
    results[name] = cv
    print(f'{name}: CV acc = {cv:.4f} ± {std:.4f}  ->  {path}')
    return sub

## Бейзлайны

In [5]:
from sklearn.dummy import DummyClassifier

_ = make_submission(DummyClassifier(strategy='most_frequent'), 'majority')

majority: CV acc = 0.3055 ± 0.0000  ->  submissions/sub_majority_cv0.3055.csv


In [6]:
from sklearn.base import BaseEstimator, ClassifierMixin

class EntityPrior(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.default_ = y.mode()[0]
        self.map_ = y.groupby(X.values).agg(lambda s: s.mode()[0])
        return self

    def predict(self, X):
        return X.map(self.map_).fillna(self.default_).values

_ = make_submission(EntityPrior(), 'entity_prior', X=train_df['Entity'], X_test=test_df['Entity'])

entity_prior: CV acc = 0.4161 ± 0.0051  ->  submissions/sub_entity_prior_cv0.4161.csv


In [7]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import make_pipeline, make_union

model = make_pipeline(
    CountVectorizer(ngram_range=(1, 2), min_df=2),
    ComplementNB(),
)
_ = make_submission(model, 'nb_word')

nb_word: CV acc = 0.5403 ± 0.0044  ->  submissions/sub_nb_word_cv0.5403.csv


In [8]:
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(C=1, max_iter=1000),
)
_ = make_submission(model, 'tfidf_lr')

tfidf_lr: CV acc = 0.5787 ± 0.0046  ->  submissions/sub_tfidf_lr_cv0.5787.csv


In [9]:
from sklearn.svm import LinearSVC

model = make_pipeline(
    make_union(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3,
                        sublinear_tf=True, max_features=300_000),
    ),
    LinearSVC(C=0.2),
)
_ = make_submission(model, 'tfidf_wordchar_svc')

tfidf_wordchar_svc: CV acc = 0.5812 ± 0.0033  ->  submissions/sub_tfidf_wordchar_svc_cv0.5812.csv


In [10]:
tr_idx, va_idx = folds[0]
model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
pd.crosstab(y.iloc[va_idx], model.predict(X.iloc[va_idx]), normalize='index').round(2)

col_0,Irrelevant,Negative,Neutral,Positive
Sentiment,,,,
Irrelevant,0.29,0.27,0.19,0.25
Negative,0.05,0.73,0.10,0.11
Neutral,0.08,0.19,0.54,0.18
Positive,0.07,0.15,0.15,0.63


## Эмбеддинги MiniLM

In [11]:
from sentence_transformers import SentenceTransformer
import torch

EMB_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
os.makedirs('emb_cache', exist_ok=True)

def embed(texts, tag):
    path = f'emb_cache/{EMB_MODEL.split("/")[-1]}_{tag}.npy'
    if os.path.exists(path):
        return np.load(path)
    device = 'mps' if torch.backends.mps.is_available() else 'cpu'
    st = SentenceTransformer(EMB_MODEL, device=device)
    emb = st.encode(texts.tolist(), batch_size=256, normalize_embeddings=True, show_progress_bar=True)
    np.save(path, emb)
    return emb

E = embed(X, 'train')
E_test = embed(X_test, 'test')
_ = make_submission(LogisticRegression(C=10, max_iter=2000), 'minilm_lr', X=E, X_test=E_test)

minilm_lr: CV acc = 0.5695 ± 0.0065  ->  submissions/sub_minilm_lr_cv0.5695.csv


In [12]:
from sklearn.compose import ColumnTransformer

emb_cols = [f'e{i}' for i in range(E.shape[1])]
D = pd.concat([X.rename('text'), pd.DataFrame(E, columns=emb_cols)], axis=1)
D_test = pd.concat([X_test.rename('text'), pd.DataFrame(E_test, columns=emb_cols)], axis=1)

model = make_pipeline(
    ColumnTransformer([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), 'text'),
        ('emb', 'passthrough', emb_cols),
    ]),
    LogisticRegression(C=1, max_iter=2000),
)
_ = make_submission(model, 'tfidf_minilm_lr', X=D, X_test=D_test)

tfidf_minilm_lr: CV acc = 0.6001 ± 0.0041  ->  submissions/sub_tfidf_minilm_lr_cv0.6001.csv


In [13]:
pd.Series(results, name='cv_acc').sort_values(ascending=False).round(4).to_frame()

,cv_acc
tfidf_minilm_lr,0.6001
tfidf_wordchar_svc,0.5812
tfidf_lr,0.5787
minilm_lr,0.5695
nb_word,0.5403
entity_prior,0.4161
majority,0.3055


## Трансформеры

Обучены на Kaggle GPU: `roberta_kaggle.ipynb`, `deberta_kaggle.ipynb`, `deberta_large_kaggle.ipynb` (фолд 0), `deberta_large_folds_kaggle.ipynb` (фолды 1–4). Вероятности лежат в `preds/`.

In [14]:
classes = np.array(sorted(LABELS))
y_arr = y.to_numpy()

for f in sorted(glob.glob('preds/debertaL_f*_val.npy')):
    k = f.split('_f')[1][0]
    va = np.load(f'preds/debertaL_f{k}_va_idx.npy')
    print(f'debertaL fold {k}: val acc = {(classes[np.load(f).argmax(1)] == y_arr[va]).mean():.4f}')

debertaL fold 0: val acc = 0.6887


In [15]:
va = np.load('preds/debertaL_f0_va_idx.npy')
pred = classes[np.load('preds/debertaL_f0_val.npy').argmax(1)]
pd.crosstab(y_arr[va], pred, rownames=['true'], colnames=['pred'], normalize='index').round(2)

pred,Irrelevant,Negative,Neutral,Positive
true,,,,
Irrelevant,0.59,0.10,0.14,0.17
Negative,0.07,0.76,0.09,0.07
Neutral,0.11,0.14,0.61,0.15
Positive,0.07,0.08,0.12,0.73


In [16]:
test_L = np.mean([np.load(f) for f in sorted(glob.glob('preds/debertaL_f*_test.npy'))], axis=0)
test_B = np.load('preds/deberta_test.npy')
test_R = np.load('preds/roberta_test.npy')

proba = 0.5 * test_L + 0.3 * test_B + 0.2 * test_R
sub = pd.DataFrame({'ID': test_df['ID'], 'Sentiment': classes[proba.argmax(1)]})
sub.to_csv(f'{SUB_DIR}/sub_ensemble.csv', index=False)
sub['Sentiment'].value_counts(normalize=True).round(3)

Sentiment
Positive      0.312
Neutral       0.268
Negative      0.263
Irrelevant    0.157
Name: proportion, dtype: float64